# Imports and paths

In [2]:
# imports and paths
from pathlib import Path
from cosipy.response import RspConverter
import numpy as np
import gzip

response_dir = Path(".")
rsp_file_name = "ResponseContinuum.o4.e100_10000.b10log.s10396905069491.m166.filtered.nonsparse.binnedimaging.imagingresponse.rsp.gz"
rsp_file_path = response_dir / rsp_file_name


h5_file_name = "ResponseContinuum.o4.e100_10000.b10log.s10396905069491.m166.filtered.nonsparse.binnedimaging.imagingresponse_nside16.area.h5"
h5_file_path = response_dir / h5_file_name

# Determine the count data type

To avoid repeating the dtype-detection scan in future conversions, first determine the smallest integer data type capable of storing the count in every response bin.

If the required dtype is already known for this exact RSP file, this step can be skipped. Otherwise, run the probe below and record the resulting dtype for future conversions.

> Note: The initial probe still reads the entire RSP file once. The first probe-plus-conversion workflow therefore reads the file twice in total. Subsequent conversions can skip the probe and read the RSP only once by explicitly passing the recorded dtype as `elt_type`.

In [ ]:
converter = RspConverter(
    norm="Linear",
    norm_params=[10, 10000],
    quiet=False,              # False 表示启用进度条
    bufsize=50_000_000,
)

dtype = converter._get_min_elt_type(rsp_file_path)

print("Required dtype:", dtype)

# Estimate the RAM required for RSP conversion from the response header

In [3]:
# Set this to the dtype obtained from the probe, e.g. np.uint8.
# Use None if the required dtype is still unknown.
known_dtype = np.uint8

# Use the same buffer size planned for RspConverter.
bufsize = 50_000_000

# A heuristic allowance for Python, NumPy, parsing, HDF5, and compression
# buffers. The converter does not require two full copies of the response.
planning_factor = 1.25


def read_rsp_header(rsp_path):
    """Read the RSP header and return its text and total number of bins."""

    rsp_path = Path(rsp_path)
    opener = gzip.open if rsp_path.suffix.lower() == ".gz" else open

    header_lines = []
    nbins = None

    with opener(rsp_path, "rt") as file:
        for line in file:
            header_lines.append(line.rstrip())

            fields = line.split()
            if fields and fields[0] == "StartStream":
                nbins = int(fields[1])
                break

    if nbins is None:
        raise ValueError("Could not find 'StartStream' in the RSP header.")

    return "\n".join(header_lines), nbins


def to_gib(nbytes):
    """Convert bytes to GiB."""

    return nbytes / 2**30


header, nbins = read_rsp_header(rsp_file_path)

print(f"RSP file: {rsp_file_path}")
print(f"Number of response bins: {nbins:,}")
print()

if known_dtype is not None:
    # Case 1: the dtype is already known
    dtype = np.dtype(known_dtype)
    counts_bytes = nbins * dtype.itemsize
    planning_bytes = counts_bytes * planning_factor

    print("Known-dtype estimate")
    print("--------------------")
    print(f"Count dtype: {dtype.name}")
    print(f"Bytes per bin: {dtype.itemsize}")
    print(f"Theoretical counts-array RAM: {to_gib(counts_bytes):.2f} GiB")
    print(
        f"Planning estimate with {planning_factor:.0%} allowance: "
        f"{to_gib(planning_bytes):.2f} GiB"
    )
    print()
    print(
        "Round this estimate up to the next memory tier available in "
        "the Jupyter Interactive App."
    )

else:
    # Case 2: the dtype is unknown
    print("Unknown-dtype estimate")
    print("----------------------")
    print(
        "The RSP header contains the number of bins, but not the maximum "
        "count. Therefore, the required dtype cannot be determined from "
        "the header alone."
    )
    print()
    print(f"{'Possible dtype':<16}{'Counts RAM':>15}{'Planning estimate':>22}")
    print("-" * 53)

    for candidate in (np.uint8, np.uint16, np.uint32, np.uint64):
        dtype = np.dtype(candidate)
        counts_bytes = nbins * dtype.itemsize
        planning_bytes = counts_bytes * planning_factor

        print(
            f"{dtype.name:<16}"
            f"{to_gib(counts_bytes):>12.2f} GiB"
            f"{to_gib(planning_bytes):>19.2f} GiB"
        )

    # During the dtype probe, only bounded text and uint64 parsing buffers
    # are retained. This is a conservative buffer-only estimate and does
    # not include the Python environment itself.
    probe_buffer_bytes = 10 * bufsize

    print()
    print(
        "Approximate upper estimate for dtype-probe working buffers: "
        f"{to_gib(probe_buffer_bytes):.2f} GiB"
    )
    print(
        "A 4–8 GiB allocation is normally sufficient for the dtype probe; "
        "the full conversion requires the RAM shown in the table."
    )

RSP file: ResponseContinuum.o4.e100_10000.b10log.s10396905069491.m166.filtered.nonsparse.binnedimaging.imagingresponse.rsp.gz
Number of response bins: 56,623,104,000

Known-dtype estimate
--------------------
Count dtype: uint8
Bytes per bin: 1
Theoretical counts-array RAM: 52.73 GiB
Planning estimate with 125% allowance: 65.92 GiB

Round this estimate up to the next memory tier available in the Jupyter Interactive App.


# Convert the response

In [5]:
converter = RspConverter(
    norm="Linear",               # 只在 RSP 缺少 SP 时使用
    norm_params=[10, 10000],     # 只在 RSP 缺少 SP 时使用
    quiet=False,
    bufsize=50_000_000,
)

output = converter.convert_to_h5(
    rsp_filename=rsp_file_path,
    h5_filename=h5_file_path,
    overwrite=False,
    compress=True,
    elt_type=np.uint8,  # 使用已经探测到的结果
    pa_convention=None,
)

print(output)

Reading counts:   0%|          | 0/56623104000 [00:00<?, ?it/s]

Writing chunks:   0%|          | 0/30720 [00:00<?, ?it/s]

ResponseContinuum.o4.e100_10000.b10log.s10396905069491.m166.filtered.nonsparse.binnedimaging.imagingresponse_nside16.area.h5


# Valid and check axes

In [8]:
import h5py as h5
import hdf5plugin

with h5.File(h5_file_path, "r") as f:
    counts = f["DRM/COUNTS"]
    properties = counts.id.get_create_plist()

    print("Number of filters:", properties.get_nfilters())

    for index in range(properties.get_nfilters()):
        filter_id, flags, parameters, name = properties.get_filter(index)

        if isinstance(name, bytes):
            name = name.decode(errors="replace")

        print("Filter ID:", filter_id)
        print("Filter name:", name)
        print("Filter parameters:", parameters)

Number of filters: 1
Filter ID: 32008
Filter name: bitshuffle; see https://github.com/kiyo-masui/bitshuffle
Filter parameters: (0, 4, 1, 0, 2)


In [9]:
from cosipy.response import FullDetectorResponse

with FullDetectorResponse.open(h5_file_path) as response:
    print("Response shape:", response.shape)
    print("Axis labels:", response.axes.labels)
    print("Response unit:", response.unit)

    # 读取较小的测试切片，不加载完整 response
    sample_counts = response.get_counts(
        0,
        em_slice=slice(0, 1),
    )

    print("Sample counts shape:", sample_counts.shape)
    print("Sample counts dtype:", sample_counts.dtype)

Response shape: (3072, 10, 10, 60, 3072)
Axis labels: ['NuLambda' 'Ei' 'Em' 'Phi' 'PsiChi']
Response unit: cm2
Sample counts shape: (10, 1, 60, 3072)
Sample counts dtype: uint8


# Check Compression Level

In [15]:
from pathlib import Path

import h5py as h5
import hdf5plugin

h5_path = Path(h5_file_path)

with h5.File(h5_path, "r") as f:
    counts = f["DRM/COUNTS"]

    logical_bytes = counts.size * counts.dtype.itemsize
    physical_bytes = h5_path.stat().st_size

    print(f"Logical COUNTS size: {logical_bytes / 2**30:.3f} GiB")
    print(f"Physical HDF5 size: {physical_bytes / 2**30:.3f} GiB")
    print(f"Compression ratio: {logical_bytes / physical_bytes:.2f}x")
    print(
        "Reduction relative to float32 representation: "
        f"{logical_bytes * 4 / physical_bytes:.2f}x"
    )

Logical COUNTS size: 52.734 GiB
Physical HDF5 size: 4.089 GiB
Compression ratio: 12.90x
Reduction relative to float32 representation: 51.59x
